# Week 22 - Closing the Drift -> Retrain -> Redeploy Loop on MWAA

You wrote a fraud classifier in Week 19. You monitored it in Week 20. You triggered an instructor-prebuilt DAG in Week 21. Today you author the DAGs that close the loop. Three labs, three DAGs you upload to MWAA, and by the end of class your code runs the entire detect -> retrain -> redeploy cycle. The Strands supervisor calls the new endpoint and cites the model package YOU just registered.

## Learning objectives

By the end of this 2-hour session you will be able to:

1. AUTHOR and upload a drift-detection DAG that computes PSI on a categorical feature against the W20 baseline, writes a drift signal to S3, and uses `ShortCircuitOperator` (with `ignore_downstream_trigger_rules=False`) to gate downstream work while still running cleanup tasks.
2. AUTHOR a retrain-and-register DAG using `SageMakerTrainingOperator` and `SageMakerRegisterModelVersionOperator` from `apache-airflow-providers-amazon==9.0.0`, with a `conf={"use_pretrained": true}` switch so class labs reuse a pre-staged artifact instead of burning ten minutes on a real GPU training run.
3. COMPOSE a top-level DAG with `TriggerDagRunOperator` that chains Lab 1 -> Lab 2 -> a new `update_endpoint` task, polls `describe_endpoint` for `InService`, and then proves the loop closed by asking a Bedrock Haiku supervisor (single Converse call) to score a known-fraud transaction AND cite the new `ModelPackageArn` traced via `describe_endpoint -> describe_endpoint_config -> describe_model`.

## The scenario

Six months in production. Last night the PSI alarm fired. Spending patterns shifted - more crypto-exchange merchants - and the model now mis-flags legitimate transactions. The pager went off at 3am. Today you do NOT trigger a DAG that someone else wrote. You sit down in Databricks and you write the three DAGs that, by end of class, automate the next 3am incident out of existence. When the loop closes, your Bedrock supervisor calls the endpoint, gets the new model's verdict, and traces the lineage back to the model package YOUR DAG just registered.

Production-craft topics like Celery vs KEDA executor comparison, AutoDAG / dynamic DAG generation, and catchup / backfilling are intentionally deferred to the Week 24 capstone. You already saw them in the pre-class videos; today is about closing the loop.


## Environment Setup

**Platform**: Azure Databricks (Runtime 15.4 LTS, Python 3.10). Same cluster shape as Week 20 and Week 21.

The DAGs you write today RUN on MWAA (`bread-academy-airflow`, Airflow 2.10.3, `apache-airflow-providers-amazon==9.0.0`). You author them in this notebook, upload them to S3, MWAA picks them up in ~30 seconds, and you trigger them via `airflow:InvokeRestApi`. Same author-upload-poll-trigger flow you learned in Week 21.

We install `apache-airflow` and `apache-airflow-providers-amazon` LOCALLY only so import errors in your DAG strings surface here in Databricks before the file ever hits MWAA. The DAG itself executes on MWAA's runtime, not on this cluster.

**Libraries installed**:

- `numpy<2`, `pandas<2` (pinned first to protect the runtime pyarrow)
- `boto3>=1.36` (Bedrock Converse API + MWAA REST)
- `sagemaker>=2.230,<3` (v3 breaks `from sagemaker import get_execution_role`)
- `mlflow-skinny>=2.13,<3` (W19 registry compatibility)
- `requests>=2.31`
- `apache-airflow==2.10.3`
- `apache-airflow-providers-amazon==9.0.0`

Run the cell below, restart the kernel when prompted, then continue from the top.


In [ ]:
# Install pinned libraries. Safe to re-run; %pip is a Databricks magic that
# pins into the notebook-scoped Python env. After the install finishes,
# dbutils.library.restartPython() restarts the kernel so the new versions
# are picked up - then re-run from the top.
#
# We deliberately do NOT install apache-airflow or the Amazon provider here.
# The DAG files are STRING payloads we upload to MWAA via S3; the
# notebook itself never imports airflow. Installing apache-airflow on a
# Databricks cluster also pulls ~80 transitive packages (flask, celery,
# sqlalchemy ...) that fight with the DBR runtime.
%pip install --quiet \
    "numpy<2" \
    "pandas<2" \
    "boto3>=1.36" \
    "sagemaker>=2.230,<3" \
    "mlflow-skinny>=2.13,<3" \
    "requests>=2.31"

dbutils.library.restartPython()

# Verify versions (importlib.metadata, NEVER pkg.__version__).
from importlib.metadata import version
for pkg in ["boto3", "sagemaker", "mlflow-skinny", "requests"]:
    try:
        print(f"{pkg:40s} {version(pkg)}")
    except Exception as e:
        print(f"{pkg:40s} NOT INSTALLED ({e})")


In [ ]:
# Standard Databricks setup - same pattern as Weeks 20 and 21.
# Pull per-student AWS keys from aws-course-creds-NN, class-wide config
# from aws-course-shared. No getpass, no sagemaker.Session(), no
# get_execution_role() - those are SageMaker Studio patterns, not
# Databricks patterns.

import os
import json
import time
import random
import boto3

# Derive this student's number from the Databricks identity.
_user = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().userName().get()
)
# BREAD_SMOKE_STUDENT_ID lets the smoke runner override the student
# identity without changing notebook code. Empty in real student runs.
_num = os.environ.get("BREAD_SMOKE_STUDENT_ID") or (
    _user.split("@")[0].split("-")[1] if _user.startswith("student-") else "01"
)
creds_scope = f"aws-course-creds-{_num}"

# Long-lived IAM-user keys (no AWS_SESSION_TOKEN; MFA is offline for this account).
AWS_ACCESS_KEY_ID     = dbutils.secrets.get(scope=creds_scope, key="aws-access-key-id")
AWS_SECRET_ACCESS_KEY = dbutils.secrets.get(scope=creds_scope, key="aws-secret-access-key")
AWS_REGION            = dbutils.secrets.get(scope="aws-course-shared", key="aws-region")

os.environ["AWS_ACCESS_KEY_ID"]     = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_REGION"]            = AWS_REGION
os.environ["AWS_DEFAULT_REGION"]    = AWS_REGION

# Boto3 clients shared across labs.
sts              = boto3.client("sts",              region_name=AWS_REGION)
s3               = boto3.client("s3",               region_name=AWS_REGION)
mwaa             = boto3.client("mwaa",             region_name=AWS_REGION)
sagemaker_client = boto3.client("sagemaker",        region_name=AWS_REGION)
sm_runtime       = boto3.client("sagemaker-runtime", region_name=AWS_REGION)
bedrock_runtime  = boto3.client("bedrock-runtime",  region_name=AWS_REGION)

# Class-wide constants.
MWAA_ENV         = "bread-academy-airflow"
DAGS_BUCKET      = "bread-academy-airflow-dags"
SHARED_BUCKET    = "bread-academy-shared"
# This class has 3 cohorts of 22 students. Each cohort gets its OWN endpoint
# the instructor deployed: fraud-classifier-endpoint-cohort-{1,2,3}.
cohort = (int(_num) - 1) // 20 + 1
ENDPOINT_NAME_BASE = dbutils.secrets.get(scope="aws-course-shared", key="endpoint-name-base")
ENDPOINT_NAME    = f"{ENDPOINT_NAME_BASE}-{cohort}"
PACKAGE_GROUP    = f"fraud-classifier-week19-student-{int(_num):02d}"
BEDROCK_MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

# Per-student identifiers. STUDENT_ID is the integer; DAG_PREFIX is the S3
# key prefix where this student's DAG files live. dag_ids include STUDENT_ID
# so 20 students in the same MWAA metadata DB do not collide.
STUDENT_ID  = int(_num)
DAG_PREFIX  = f"dags/student_{STUDENT_ID:02d}"
SAGEMAKER_ROLE_ARN = dbutils.secrets.get(scope="aws-course-shared", key="sagemaker-execution-role-arn")

print(f"Student ID:         {STUDENT_ID}")
print(f"AWS region:         {AWS_REGION}")
print(f"MWAA env:           {MWAA_ENV}")
print(f"DAG bucket prefix:  s3://{DAGS_BUCKET}/{DAG_PREFIX}/")
print(f"dag_id pattern:     week22_lab{{1,2,3}}_{STUDENT_ID}")


In [ ]:
# Pre-flight probes - fail loud NOW if anything we depend on is missing.
# Cheaper to find out here than to discover it five minutes into Lab 3.

# Probe 1: IAM keys work and we are who we think we are.
ident = sts.get_caller_identity()
print(f"Probe 1 (STS):         arn={ident['Arn']}")

# Probe 2: MWAA reachable. list_environments returns names; we just need a 200.
envs = mwaa.list_environments()["Environments"]
assert MWAA_ENV in envs, (
    f"MWAA env {MWAA_ENV} not visible to this role. "
    "Ask your instructor to check IAM permissions."
)
print(f"Probe 2 (MWAA):        env {MWAA_ENV} reachable")

# Probe 3: shared assets exist where the DAGs will look for them.
for key in [
    "drift-baselines/merchant_country_baseline.json",
    "pretrained/model.tar.gz",
    "pretrained/inference_image_uri.txt",
    "sample/recent_transactions.csv",
]:
    try:
        s3.head_object(Bucket=SHARED_BUCKET, Key=key)
        print(f"Probe 3 (S3):          s3://{SHARED_BUCKET}/{key} ok")
    except Exception as e:
        print(f"Probe 3 (S3):          MISSING s3://{SHARED_BUCKET}/{key} -> ask instructor")
        raise

# Probe 4: Bedrock Haiku 3 access. 10-token converse, fail loud.
try:
    bedrock_runtime.converse(
        modelId=BEDROCK_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": "ping"}]}],
        inferenceConfig={"maxTokens": 10, "temperature": 0},
    )
    print(f"Probe 4 (Bedrock):     {BEDROCK_MODEL_ID} ok")
except Exception:
    print(f"Ask your instructor to enable Bedrock access for {BEDROCK_MODEL_ID}")
    raise

print("\nAll pre-flight probes passed. You are clear to author DAGs.")


## Recap: the W21 author-upload-poll-trigger flow

In Week 21 you wrote DAG files as Python strings inside this notebook, uploaded them to S3, waited for MWAA to pick them up, and triggered them via `airflow:InvokeRestApi`. That flow is the spine of this week too - we just author harder DAGs on top of it.

The four helper functions you used in W21 are reproduced in the next cell so this notebook is self-contained:

- `upload_dag(student_id, lab_number, dag_python_string)` writes the DAG file to `s3://bread-academy-airflow-dags/dags/student_NN/lab_N.py` via `s3.put_object`.
- `wait_for_dag_pickup(dag_id, timeout=180)` polls MWAA's REST `/dags/{dag_id}` endpoint until it returns 200, then returns. Raises on timeout.
- `trigger_dag(dag_id, conf=None)` POSTs to `/dags/{dag_id}/dagRuns` with a fresh `dag_run_id` and an optional `conf` dict, returns the run id.
- `wait_for_dag_run(dag_id, run_id, timeout=900)` polls `/dags/{dag_id}/dagRuns/{run_id}` until state is `success` or `failed`. Raises on `failed` or timeout.

**Crucial reminder**: ALWAYS overwrite the same filename per lab. MWAA picks up updates to an existing file in about 30 seconds; brand-new files take about 5 minutes (governed by `scheduler.dag_dir_list_interval`). Re-using `lab_1.py` keeps every iteration on the 30-second path. Do NOT save your second attempt as `lab_1_v2.py` - that will cost you 5 minutes of class time per iteration.

Per-student S3 prefix is a soft convention so the 20 of you do not overwrite each other's DAG files. `dag_id` also carries `STUDENT_ID` so no two DAGs collide in MWAA's metadata DB either.


In [ ]:
# Author-upload-poll-trigger helpers. Same shape as Week 21; we paste
# them here so the notebook is self-contained. All four use the
# mwaa.invoke_rest_api boto3 call, which signs requests with SigV4 under
# the hood - no separate web-login-token plumbing required.

def upload_dag(student_id: int, lab_number: int, dag_python_string: str) -> str:
    """Upload a DAG string to s3://bread-academy-airflow-dags/dags/student_NN/lab_N.py.

    Always overwrites the SAME filename per lab so MWAA's 30-second
    update sync applies (not the 5-minute new-DAG sync).
    """
    key = f"dags/student_{student_id:02d}/lab_{lab_number}.py"
    s3.put_object(
        Bucket=DAGS_BUCKET,
        Key=key,
        Body=dag_python_string.encode("utf-8"),
        ContentType="text/x-python",
    )
    uri = f"s3://{DAGS_BUCKET}/{key}"
    print(f"upload_dag: wrote {uri} ({len(dag_python_string)} bytes)")
    return uri


def wait_for_dag_pickup(dag_id: str, timeout: int = 180) -> None:
    """Poll MWAA REST /dags/{dag_id} until it returns 200 or we time out."""
    print(f"wait_for_dag_pickup: polling for {dag_id} (timeout {timeout}s)")
    deadline = time.time() + timeout
    while time.time() < deadline:
        resp = mwaa.invoke_rest_api(
            Name=MWAA_ENV,
            Method="GET",
            Path=f"/dags/{dag_id}",
        )
        if resp.get("RestApiStatusCode") == 200:
            print(f"wait_for_dag_pickup: {dag_id} registered")
            return
        time.sleep(5)
    raise RuntimeError(
        f"{dag_id} did not appear within {timeout}s. Check MWAA UI -> "
        "DAG Import Errors panel for syntax issues."
    )


def unpause_dag(dag_id: str) -> None:
    """Unpause a DAG via PATCH /dags/{dag_id}. MWAA creates new DAGs PAUSED, so
    a trigger does nothing until the DAG is unpaused. Idempotent - safe to call
    every time."""
    resp = mwaa.invoke_rest_api(
        Name=MWAA_ENV,
        Method="PATCH",
        Path=f"/dags/{dag_id}",
        Body={"is_paused": False},
    )
    if resp["RestApiStatusCode"] not in (200,):
        raise RuntimeError(f"unpause_dag failed for {dag_id}: {resp}")
    print(f"Unpaused {dag_id}")


def trigger_dag(dag_id: str, conf: dict = None) -> str:
    """POST /dags/{dag_id}/dagRuns and return the new run_id."""
    run_id = f"student-{STUDENT_ID:02d}-{int(time.time())}"
    unpause_dag(dag_id)  # MWAA creates DAGs paused; unpause before triggering
    body = {"dag_run_id": run_id}
    if conf is not None:
        body["conf"] = conf
    resp = mwaa.invoke_rest_api(
        Name=MWAA_ENV,
        Method="POST",
        Path=f"/dags/{dag_id}/dagRuns",
        Body=body,
    )
    if resp.get("RestApiStatusCode") not in (200, 201):
        raise RuntimeError(f"trigger_dag failed: {resp}")
    print(f"trigger_dag: {dag_id} run_id={run_id}")
    return run_id


def wait_for_dag_run(dag_id: str, run_id: str, timeout: int = 900) -> str:
    """Poll /dags/{dag_id}/dagRuns/{run_id} until state is success or failed."""
    print(f"wait_for_dag_run: polling {dag_id}/{run_id} (timeout {timeout}s)")
    deadline = time.time() + timeout
    while time.time() < deadline:
        resp = mwaa.invoke_rest_api(
            Name=MWAA_ENV,
            Method="GET",
            Path=f"/dags/{dag_id}/dagRuns/{run_id}",
        )
        state = (resp.get("RestApiResponse") or {}).get("state")
        if state in ("success", "failed"):
            print(f"wait_for_dag_run: {dag_id}/{run_id} -> {state}")
            if state == "failed":
                raise RuntimeError(f"{dag_id}/{run_id} failed - check the Airflow UI task logs")
            return state
        time.sleep(10)
    raise RuntimeError(f"{dag_id}/{run_id} did not finish within {timeout}s")


print("Helpers defined: upload_dag, wait_for_dag_pickup, trigger_dag, wait_for_dag_run")


## Lab 1 - Drift-detection DAG (theory)

**Why a drift DAG belongs at the top of the loop.** In Week 20 you wrote a drift detector that lived in a notebook. That is fine for an investigation, useless for production. Production needs the detector on a schedule, with retries, with visibility, and with a clean signal that downstream automation can read. That is an Airflow DAG.

**PSI in 60 seconds.** The Population Stability Index measures how much a categorical (or binned numerical) distribution has shifted from a baseline:

```
PSI = sum_i (O_i - E_i) * ln(O_i / E_i)
```

where `O_i` is the observed percentage in category `i` today, and `E_i` is the expected percentage from the baseline. Industry convention: `PSI < 0.1` is stable, `0.1 <= PSI <= 0.25` is moderate drift (investigate), `PSI > 0.25` is severe drift (act). We pick `0.2` as the gate threshold to balance signal against alert fatigue - low enough to catch real shifts, high enough that one noisy day does not page the on-call engineer.

**ShortCircuit + cleanup, a footgun.** `ShortCircuitOperator` evaluates a Python callable and, on `False`, marks all downstream tasks as `skipped`. The default `ignore_downstream_trigger_rules=True` is helpful for most cases - skipping a single branch - but it ALSO silently skips a cleanup task that you wrote with `trigger_rule="all_done"`. That is almost never what you want. Set `ignore_downstream_trigger_rules=False` so cleanup ALWAYS runs even when the gate skips the retrain path. We will exploit this in Lab 1's `cleanup` task.


In [ ]:
# Live demo: compute PSI on a synthetic merchant_country distribution.
# This is the SAME function the Lab 1 DAG will call inside compute_psi -
# we run it standalone first so you see the math before you embed it in
# a DAG.

import math
import pandas as pd

# Baseline distribution (what Week 20 saved to s3://bread-academy-shared/
# drift-baselines/merchant_country_baseline.json). Today's "current"
# distribution is shifted: more "CRYPTO" merchants, fewer "RETAIL".
baseline = {"US_RETAIL": 0.50, "US_CRYPTO": 0.05, "EU_RETAIL": 0.30, "OTHER": 0.15}
current  = {"US_RETAIL": 0.35, "US_CRYPTO": 0.25, "EU_RETAIL": 0.30, "OTHER": 0.10}


def compute_psi(observed: dict, expected: dict, eps: float = 1e-6) -> float:
    """PSI = sum((O - E) * ln(O / E)) across category bins."""
    psi = 0.0
    rows = []
    for cat in expected:
        # eps prevents log(0) when a category disappears in either side.
        o = max(observed.get(cat, 0.0), eps)
        e = max(expected.get(cat, 0.0), eps)
        contrib = (o - e) * math.log(o / e)
        psi += contrib
        rows.append({"category": cat, "expected": e, "observed": o, "contribution": contrib})
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    return psi


psi_value = compute_psi(current, baseline)
print(f"\nTotal PSI = {psi_value:.4f}")
print(f"Gate (threshold 0.2): {'DRIFT - retrain' if psi_value > 0.2 else 'stable - skip'}")


## Lab 1 - Author the drift-detection DAG (~15 min)

You will write the DAG as a Python string in the next cell, upload it via `upload_dag(STUDENT_ID, 1, dag_string)`, wait for MWAA to register it, and trigger it twice (once with `force_drift=false` to see the short-circuit path, once with `force_drift=true` to see the full path).

**DAG topology** (5 tasks):

1. `pull_recent_transactions` (PythonOperator): read 1000 rows of `merchant_country` from `s3://bread-academy-shared/sample/recent_transactions.csv` (the instructor pre-staged a Spark export there so the DAG does not need a JDBC connection). Push the `value_counts(normalize=True).to_dict()` as XCom.
2. `compute_psi` (PythonOperator): load `s3://bread-academy-shared/drift-baselines/merchant_country_baseline.json`, compute PSI against the upstream XCom distribution, push the PSI float as XCom. If `dag_run.conf.get("force_drift", False)`, push a PSI of `0.99` instead - this is the override switch you use for the second trigger so the gate flips.
3. `gate_on_drift` (ShortCircuitOperator with `ignore_downstream_trigger_rules=False`): returns `True` if `psi > 0.2`, else `False`.
4. `write_drift_signal` (PythonOperator): write `{"psi": float, "gate": bool, "timestamp": iso, "student_id": int}` JSON to `s3://bread-academy-shared/drift-signals/student_NN/latest.json`. Only runs when the gate is True.
5. `cleanup` (PythonOperator with `trigger_rule="all_done"`): logs "drift run finished, gate=...", emits a heartbeat. Thanks to `ignore_downstream_trigger_rules=False` on the gate, this cleanup task runs whether the gate was True or False.

**DAG defaults**:
- `dag_id = f"week22_lab1_{STUDENT_ID}"`
- `schedule = None` (we trigger by hand)
- `catchup = False`
- `start_date = days_ago(1)`
- `default_args = {"retries": 1, "retry_delay": timedelta(minutes=1)}`

**Steps**:

1. Build the `dag_string` in the next cell.
2. `upload_dag(STUDENT_ID, 1, dag_string)`.
3. `wait_for_dag_pickup(f"week22_lab1_{STUDENT_ID}")`.
4. `trigger_dag(f"week22_lab1_{STUDENT_ID}", conf={"force_drift": False})` then `wait_for_dag_run`.
5. `trigger_dag(...)` again with `conf={"force_drift": True}` and `wait_for_dag_run`.
6. Open MWAA UI in a second browser tab - confirm the two runs are visible, gate task colored skipped/success accordingly, and the S3 signal file appears only on the `force_drift=True` run.


In [ ]:
# SOLUTION: Lab 1 - drift-detection DAG. dag_string is built as an
# f-string so STUDENT_ID, SHARED_BUCKET, and the per-student key are
# baked into the DAG file MWAA will load. Each task is a top-level
# function so the DAG file imports clean on MWAA's scheduler.

dag_string = f'''
from __future__ import annotations
import json
import math
from datetime import datetime, timedelta, timezone

import boto3
import pandas as pd
from airflow import DAG
from airflow.operators.python import PythonOperator, ShortCircuitOperator
from airflow.utils.dates import days_ago

STUDENT_ID = {STUDENT_ID}
SHARED_BUCKET = "{SHARED_BUCKET}"
BASELINE_KEY = "drift-baselines/merchant_country_baseline.json"
SAMPLE_KEY = "sample/recent_transactions.csv"
SIGNAL_KEY = f"drift-signals/student_{{STUDENT_ID:02d}}/latest.json"

default_args = {{
    "owner": f"student_{{STUDENT_ID:02d}}",
    "retries": 1,
    "retry_delay": timedelta(minutes=1),
}}


def _s3():
    return boto3.client("s3")


def pull_recent_transactions(**context):
    # Read the pre-staged 1000-row sample. value_counts(normalize=True)
    # gives us the current category percentages we feed into PSI.
    obj = _s3().get_object(Bucket=SHARED_BUCKET, Key=SAMPLE_KEY)
    df = pd.read_csv(obj["Body"])
    dist = df["merchant_country"].value_counts(normalize=True).to_dict()
    return dist


def compute_psi(**context):
    # If the caller forces drift (conf={{"force_drift": True}}), return a
    # large PSI so the gate flips True. Otherwise compute PSI against
    # the W20 baseline. eps prevents log(0) when a category is missing.
    if (context["dag_run"].conf or {{}}).get("force_drift", False):
        return 0.99
    observed = context["ti"].xcom_pull(task_ids="pull_recent_transactions")
    obj = _s3().get_object(Bucket=SHARED_BUCKET, Key=BASELINE_KEY)
    expected = json.loads(obj["Body"].read())
    eps = 1e-6
    psi = 0.0
    for cat in expected:
        o = max(observed.get(cat, 0.0), eps)
        e = max(expected.get(cat, 0.0), eps)
        psi += (o - e) * math.log(o / e)
    return float(psi)


def gate(**context):
    # ShortCircuit returns True (continue) or False (skip downstream).
    psi = context["ti"].xcom_pull(task_ids="compute_psi")
    return psi is not None and psi > 0.2


def write_signal(**context):
    # Only reached when the gate said True. Writes a JSON signal Lab 3
    # can read later if it wants to inspect the drift event.
    psi = context["ti"].xcom_pull(task_ids="compute_psi")
    body = {{
        "psi": float(psi),
        "gate": True,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "student_id": STUDENT_ID,
    }}
    _s3().put_object(Bucket=SHARED_BUCKET, Key=SIGNAL_KEY,
                     Body=json.dumps(body).encode("utf-8"),
                     ContentType="application/json")


def cleanup(**context):
    # trigger_rule="all_done" + ignore_downstream_trigger_rules=False on
    # the gate above means cleanup ALWAYS runs - True OR False gate.
    psi = context["ti"].xcom_pull(task_ids="compute_psi")
    print(f"drift run finished, psi={{psi}}, student={{STUDENT_ID}}")


with DAG(
    dag_id=f"week22_lab1_{{STUDENT_ID}}",
    description="Drift detection DAG - Week 22 Lab 1",
    default_args=default_args,
    start_date=days_ago(1),
    schedule=None,
    catchup=False,
    tags=["week22", "lab1", f"student_{{STUDENT_ID:02d}}"],
) as dag:
    t_pull  = PythonOperator(task_id="pull_recent_transactions",
                             python_callable=pull_recent_transactions)
    t_psi   = PythonOperator(task_id="compute_psi", python_callable=compute_psi)
    # Set ignore_downstream_trigger_rules=False so cleanup STILL runs
    # when the gate skips write_drift_signal. The default True would
    # silently skip cleanup too - a classic footgun.
    t_gate  = ShortCircuitOperator(task_id="gate_on_drift", python_callable=gate,
                                   ignore_downstream_trigger_rules=False)
    t_write = PythonOperator(task_id="write_drift_signal", python_callable=write_signal)
    t_cleanup = PythonOperator(task_id="cleanup", python_callable=cleanup,
                               trigger_rule="all_done")
    t_pull >> t_psi >> t_gate >> t_write >> t_cleanup
'''

print(f"Lab 1 dag_string built ({len(dag_string)} chars)")


In [ ]:
# Lab 1 - upload, poll, trigger twice, observe.
# First trigger uses force_drift=False so the ShortCircuit gate evaluates
# the real PSI (small, below 0.2) and the write_drift_signal task is
# skipped while cleanup STILL runs (because we set
# ignore_downstream_trigger_rules=False on the gate).
# Second trigger uses force_drift=True so the gate flips True, the
# signal lands in S3, and cleanup runs again.

lab1_dag_id = f"week22_lab1_{STUDENT_ID}"

upload_dag(STUDENT_ID, 1, dag_string)
wait_for_dag_pickup(lab1_dag_id)

# Run A: real PSI (probably below 0.2) - watch the gate skip the write task.
run_a = trigger_dag(lab1_dag_id, conf={"force_drift": False})
wait_for_dag_run(lab1_dag_id, run_a, timeout=300)

# Run B: forced drift - watch the signal file appear in S3.
run_b = trigger_dag(lab1_dag_id, conf={"force_drift": True})
wait_for_dag_run(lab1_dag_id, run_b, timeout=300)

# Confirm the drift signal file exists after Run B.
signal_key = f"drift-signals/student_{STUDENT_ID:02d}/latest.json"
obj = s3.get_object(Bucket=SHARED_BUCKET, Key=signal_key)
print("\nDrift signal contents:")
print(obj["Body"].read().decode("utf-8"))


## Think About It - Lab 1 reflection

Two questions to roll over in your head before Lab 2:

1. The ShortCircuit gate is on a Python boolean returned by `compute_psi`. What happens if `compute_psi` itself fails - a network blip reading the baseline JSON, a malformed CSV row, an unexpected category in `merchant_country`? Does the `cleanup` task still run? Why? (Hint: `trigger_rule="all_done"` on cleanup and how upstream FAILURE propagates is different from upstream SKIPPED.)

2. We chose `0.2` as the PSI threshold. A risk team might prefer `0.1` (more sensitive, more false alarms). What changes DOWNSTREAM when you drop the threshold? Remember that every gate-true event will, by end of Lab 3, trigger a SageMaker training job and an endpoint update across the whole class.


## Lab 2 - Retrain + register DAG (theory)

**Why the retrain step is a DAG, not a script.** Once Lab 1 says "drift", the next stage has to be repeatable, observable, and idempotent. A bash script kicked off from CloudWatch alarms is fine for week one of a hackathon; for a real production loop you want retries, lineage, scheduling, and a single pane of glass. That is Airflow.

**`SageMakerTrainingOperator` vs raw boto3.** The operator wraps `create_training_job` and gives you:

- `wait_for_completion=True` plus a sensor-like internal poll with `check_interval` and `max_ingestion_time`.
- `check_if_job_exists=True` (default) plus `action_if_job_exists='timestamp'` (default) - so re-running the DAG on the same day appends a timestamp to the job name instead of crashing. That is the rerun-safety we want.
- `print_log=True` tails CloudWatch logs into the Airflow task log.

In exchange, you hand the operator the same `config` dict you would have handed `create_training_job`. The boto3-level payload is still visible to you - the operator does not hide it.

**The `use_pretrained` switch.** A real GPU training job for the fraud classifier takes 10+ minutes. Twenty students times ten minutes is your entire class period. We add a `BranchPythonOperator` that reads `dag_run.conf.get("use_pretrained", True)`. If True (the class default), the branch jumps to a 1-second `skip_to_register` task that emits the pre-staged `s3://bread-academy-shared/pretrained/model.tar.gz` URI as XCom. If False, the real `SageMakerTrainingOperator` runs. Either way, the registration task downstream pulls `model_url` from XCom - same registration code path for both branches.

**`SageMakerRegisterModelVersionOperator` (provider 9.0.0).** This is the canonical "register a model package version" operator on the Amazon provider. `SageMakerCreateModelPackageOperator` does NOT exist - if you grep for it in the provider source you will not find it. The right operator takes `image_uri`, `model_url`, `package_group_name`, optional `package_desc`, and a `model_approval` string (we will pass `"Approved"` for class speed so the model package is immediately consumable; in production this would be `"PendingManualApproval"`).


In [ ]:
# Live demo: construct (do NOT execute) a SageMakerRegisterModelVersionOperator
# so you can see the constructor up close before pasting it into a DAG.
# We construct it OUTSIDE a DAG context just to inspect its fields - in a
# real DAG you would place this inside a `with DAG(...)` block.

from airflow.providers.amazon.aws.operators.sagemaker import (
    SageMakerRegisterModelVersionOperator,
)

# Read the pre-staged inference image URI the instructor wrote at
# s3://bread-academy-shared/pretrained/inference_image_uri.txt
img_uri_obj = s3.get_object(
    Bucket=SHARED_BUCKET, Key="pretrained/inference_image_uri.txt"
)
INFERENCE_IMAGE_URI = img_uri_obj["Body"].read().decode("utf-8").strip()
PRETRAINED_MODEL_URL = f"s3://{SHARED_BUCKET}/pretrained/model.tar.gz"

op = SageMakerRegisterModelVersionOperator(
    task_id="register",
    image_uri=INFERENCE_IMAGE_URI,
    model_url=PRETRAINED_MODEL_URL,
    package_group_name=PACKAGE_GROUP,
    package_desc=f"Week 22 retrain by student {STUDENT_ID}",
    # Plain string is accepted; the ApprovalStatus enum from
    # airflow.providers.amazon.aws.utils.sagemaker also works.
    # "Approved" auto-promotes; production would use "PendingManualApproval".
    model_approval="Approved",
)

print("image_uri:         ", op.image_uri)
print("model_url:         ", op.model_url)
print("package_group_name:", op.package_group_name)
print("model_approval:    ", op.model_approval)
print("task_id:           ", op.task_id)


## Lab 2 - Author the retrain + register DAG (~20 min)

**DAG topology** (6 tasks):

1. `read_config` (PythonOperator): pulls `use_pretrained` from `dag_run.conf`, default True. Pushes the boolean as XCom.
2. `branch_on_pretrained` (BranchPythonOperator): returns the string `"skip_to_register"` if `use_pretrained` is True, else `"start_training"`.
3. `skip_to_register` (PythonOperator): returns `PRETRAINED_MODEL_URL` as the XCom value the downstream `register` task will pull as `model_url`.
4. `start_training` (SageMakerTrainingOperator): a 2-minute toy `create_training_job` config. Class default is to NEVER reach this task because `use_pretrained=True` is the default; if a curious student flips the switch they pay 2 minutes.
5. `register` (SageMakerRegisterModelVersionOperator): pulls `model_url` from XCom of either upstream branch (use `trigger_rule="none_failed_min_one_success"`). Registers under `fraud-classifier-week19`, auto-approves.
6. `print_arn` (PythonOperator with `trigger_rule="none_failed_min_one_success"`): pulls the `ModelPackageArn` from the register task's XCom and logs it.

**DAG defaults**:
- `dag_id = f"week22_lab2_{STUDENT_ID}"`
- `schedule = None`, `catchup = False`, `start_date = days_ago(1)`
- `default_args = {"retries": 1, "retry_delay": timedelta(minutes=1)}`

**Steps**:

1. Build the `dag_string_2` in the next cell.
2. `upload_dag(STUDENT_ID, 2, dag_string_2)`.
3. `wait_for_dag_pickup(f"week22_lab2_{STUDENT_ID}")`.
4. `trigger_dag(f"week22_lab2_{STUDENT_ID}", conf={"use_pretrained": True})`.
5. `wait_for_dag_run(...)`.
6. From this notebook, `sagemaker_client.list_model_packages(ModelPackageGroupName=PACKAGE_GROUP, SortBy="CreationTime", SortOrder="Descending", MaxResults=3)` and confirm a new version landed under your `STUDENT_ID`. Record YOUR new `ModelPackageArn` - Lab 3 will compare against it.


In [ ]:
# SOLUTION: Lab 2 - retrain + register DAG. read_config and
# branch_on_pretrained were already done; the new code is
# skip_to_register, the SageMakerTrainingOperator, the
# SageMakerRegisterModelVersionOperator pulling model_url from XCom of
# either branch via a Jinja template, print_arn, and the dependency
# chain.

dag_string_2 = f'''
from __future__ import annotations
from datetime import timedelta

from airflow import DAG
from airflow.operators.python import PythonOperator, BranchPythonOperator
from airflow.utils.dates import days_ago
from airflow.providers.amazon.aws.operators.sagemaker import (
    SageMakerTrainingOperator,
    SageMakerRegisterModelVersionOperator,
)

STUDENT_ID = {STUDENT_ID}
SHARED_BUCKET = "{SHARED_BUCKET}"
PACKAGE_GROUP = "{PACKAGE_GROUP}"
INFERENCE_IMAGE_URI = {INFERENCE_IMAGE_URI!r}
PRETRAINED_MODEL_URL = "s3://{SHARED_BUCKET}/pretrained/model.tar.gz"
SAGEMAKER_ROLE_ARN = {SAGEMAKER_ROLE_ARN!r}

default_args = {{
    "owner": f"student_{{STUDENT_ID:02d}}",
    "retries": 1,
    "retry_delay": timedelta(minutes=1),
}}


def read_config(**context):
    use_pre = (context["dag_run"].conf or {{}}).get("use_pretrained", True)
    return bool(use_pre)


def branch_on_pretrained(**context):
    use_pre = context["ti"].xcom_pull(task_ids="read_config")
    return "skip_to_register" if use_pre else "start_training"


def skip_to_register(**context):
    # Returns the pretrained tarball URI; the register task downstream
    # pulls this via XCom.
    return PRETRAINED_MODEL_URL


def print_arn(**context):
    # The register operator pushes the ModelPackageArn as XCom; pull
    # and log it so we can see lineage end-to-end in the task log.
    arn = context["ti"].xcom_pull(task_ids="register")
    print(f"ModelPackageArn registered: {{arn}}")


# Toy create_training_job config. We virtually never run this in class
# because conf={{"use_pretrained": True}} is the default.
training_config = {{
    "TrainingJobName": f"week22-lab2-student-{{STUDENT_ID:02d}}-{{{{ ts_nodash }}}}",
    "AlgorithmSpecification": {{
        "TrainingImage": INFERENCE_IMAGE_URI,
        "TrainingInputMode": "File",
    }},
    "RoleArn": SAGEMAKER_ROLE_ARN,
    "InputDataConfig": [{{
        "ChannelName": "train",
        "DataSource": {{"S3DataSource": {{
            "S3DataType": "S3Prefix",
            "S3Uri": f"s3://{SHARED_BUCKET}/training/",
            "S3DataDistributionType": "FullyReplicated",
        }}}},
    }}],
    "OutputDataConfig": {{"S3OutputPath": f"s3://{SHARED_BUCKET}/training-out/student_{{STUDENT_ID:02d}}/"}},
    "ResourceConfig": {{"InstanceType": "ml.m5.large", "InstanceCount": 1, "VolumeSizeInGB": 10}},
    "StoppingCondition": {{"MaxRuntimeInSeconds": 600}},
}}


with DAG(
    dag_id=f"week22_lab2_{{STUDENT_ID}}",
    description="Retrain + register DAG - Week 22 Lab 2",
    default_args=default_args,
    start_date=days_ago(1),
    schedule=None,
    catchup=False,
    tags=["week22", "lab2", f"student_{{STUDENT_ID:02d}}"],
) as dag:
    t_read   = PythonOperator(task_id="read_config", python_callable=read_config)
    t_branch = BranchPythonOperator(task_id="branch_on_pretrained",
                                    python_callable=branch_on_pretrained)
    t_skip   = PythonOperator(task_id="skip_to_register",
                              python_callable=skip_to_register)
    # SageMakerTrainingOperator wraps create_training_job. Defaults
    # check_if_job_exists=True + action_if_job_exists='timestamp' so
    # reruns get a timestamped name; print_log=True tails CloudWatch.
    t_train  = SageMakerTrainingOperator(
        task_id="start_training",
        config=training_config,
        wait_for_completion=True,
        print_log=True,
    )
    # The register operator pulls model_url from EITHER branch's XCom
    # via a Jinja-templated string. trigger_rule lets it run when only
    # one of the two upstream branches succeeded (the other was
    # skipped by branch_on_pretrained).
    t_reg    = SageMakerRegisterModelVersionOperator(
        task_id="register",
        image_uri=INFERENCE_IMAGE_URI,
        model_url="{{{{ ti.xcom_pull(task_ids='skip_to_register') or ti.xcom_pull(task_ids='start_training', key='Model_artifact_S3_URI') }}}}",
        package_group_name=PACKAGE_GROUP,
        package_desc=f"Week 22 retrain by student {{STUDENT_ID:02d}}",
        model_approval="Approved",
        trigger_rule="none_failed_min_one_success",
    )
    t_print  = PythonOperator(task_id="print_arn", python_callable=print_arn,
                              trigger_rule="none_failed_min_one_success")

    t_read >> t_branch >> [t_skip, t_train] >> t_reg >> t_print
'''

print(f"Lab 2 dag_string_2 built ({len(dag_string_2)} chars)")


In [ ]:
# Lab 2 - upload, poll, trigger with use_pretrained=True.
# This run skips the SageMaker training job entirely and goes straight
# to register; total wall time is dominated by the
# RegisterModelVersionOperator's create_model_package call (a few
# seconds).

lab2_dag_id = f"week22_lab2_{STUDENT_ID}"

upload_dag(STUDENT_ID, 2, dag_string_2)
wait_for_dag_pickup(lab2_dag_id)

run_lab2 = trigger_dag(lab2_dag_id, conf={"use_pretrained": True})
wait_for_dag_run(lab2_dag_id, run_lab2, timeout=300)

# Confirm a new model package version landed.
resp = sagemaker_client.list_model_packages(
    ModelPackageGroupName=PACKAGE_GROUP,
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=3,
)
print("\nLatest 3 model packages in", PACKAGE_GROUP)
for mp in resp["ModelPackageSummaryList"]:
    print(f"  v{mp['ModelPackageVersion']}: {mp['ModelPackageArn']} "
          f"approved={mp.get('ModelApprovalStatus')} created={mp['CreationTime'].isoformat()}")

# Record the latest (yours) for Lab 3 verification.
LAB2_MODEL_PACKAGE_ARN = resp["ModelPackageSummaryList"][0]["ModelPackageArn"]
print(f"\nYour Lab 2 ModelPackageArn: {LAB2_MODEL_PACKAGE_ARN}")


## Think About It - Lab 2 reflection

1. `SageMakerTrainingOperator` has `check_if_job_exists=True` by default with `action_if_job_exists='timestamp'`. What does that protect against, and what is it NOT protecting against? (Hint: it dedupes by job NAME, not by training DATA. Two retrains with the same data but different timestamped names are still two retrains.)

2. We auto-approved the model package by passing `model_approval="Approved"`. In real production you would not. Where would the human-approval gate go in this DAG, and what Airflow primitive would you use to pause until a human clicks "approve"? Think about the difference between blocking the DAG run versus emitting a notification and finishing.


## Lab 3 - Close the loop (theory)

Lab 3 is the orchestrator. It chains your Lab 1 (drift detect) and Lab 2 (retrain + register) DAGs with `TriggerDagRunOperator`, then deploys the freshly registered model, proves it is live, and tears down. One new design decision dominates this lab:

**Each student deploys their OWN endpoint, not a shared one.** The earlier instinct is to `update_endpoint` on a single class endpoint. That does not survive 60 students in one 2-hour window: SageMaker serializes updates per endpoint, so while one update is in flight every other caller gets a `ValidationException` ("endpoint is currently in Updating state"). The classic mitigations (random jitter, catch-and-retry) only thin the herd; they do not remove the contention.

The clean fix is to give every student their own endpoint:

- `create_endpoint(EndpointName=f"fraud-w22-student-NN", ...)` - a brand-new per-student endpoint. No contention, because no two students touch the same endpoint.
- `delete_endpoint(...)` as a final task with `trigger_rule="all_done"` - it runs whether or not the verification succeeded, so your endpoint never bills past the lab.

**On `create_endpoint` vs the provider operator.** The Amazon provider 9.0.0 ships `SageMakerEndpointOperator` (operation `create` or `update`), but for a one-shot per-student endpoint that we immediately delete, a plain `PythonOperator` calling `boto3.create_endpoint` / `delete_endpoint` is clearer and keeps the create and the cleanup symmetric. `retries=0` stays in `default_args`: a create that half-succeeded should fail loudly so the `all_done` cleanup still deletes the endpoint, rather than silently retrying.

In [ ]:
# Live demo: run the four-step lineage trace on the CURRENT (pre-retrain)
# endpoint. This is the same code the Lab 3 DAG's verify_new_model task
# will run; we demo it standalone here so you see all four describe
# calls before pasting them into a DAG.


def lineage_trace(endpoint_name: str) -> dict:
    """describe_endpoint -> describe_endpoint_config -> describe_model
    -> return ModelPackageArn (in PrimaryContainer.ModelPackageName)."""
    # Step 1: endpoint -> endpoint config name + status
    ep = sagemaker_client.describe_endpoint(EndpointName=endpoint_name)
    ep_config_name = ep["EndpointConfigName"]
    status = ep["EndpointStatus"]

    # Step 2: endpoint config -> first production variant's model name
    cfg = sagemaker_client.describe_endpoint_config(
        EndpointConfigName=ep_config_name
    )
    model_name = cfg["ProductionVariants"][0]["ModelName"]

    # Step 3: model -> primary container -> ModelPackageName (which
    # carries the full ModelPackageArn when the model was created
    # from a model package).
    model = sagemaker_client.describe_model(ModelName=model_name)
    model_package_arn = model["PrimaryContainer"].get("ModelPackageName")

    return {
        "endpoint_status": status,
        "endpoint_config_name": ep_config_name,
        "model_name": model_name,
        "model_package_arn": model_package_arn,
    }


current_lineage = lineage_trace(ENDPOINT_NAME)
print("Current endpoint lineage (BEFORE Lab 3 update):")
for k, v in current_lineage.items():
    print(f"  {k}: {v}")


## Lab 3 - Author the orchestrator DAG that closes the loop (~30 min)

This is the closing move. Your orchestrator DAG triggers Lab 1, waits, triggers Lab 2, waits, then deploys the new model package version to YOUR OWN per-student endpoint, polls until `InService`, asks Bedrock to score a known-fraud transaction AND cite the new model's ModelPackageArn, and finally deletes your endpoint so it does not bill forever.

**Why a per-student endpoint?** Sixty students cannot all `update_endpoint` on one shared endpoint - SageMaker serializes updates per endpoint and the rest get a `ValidationException`. Instead each student does `create_endpoint` on `fraud-w22-student-NN`, verifies on it, then `delete_endpoint` with `trigger_rule="all_done"`. No collision, no jitter hacks, no lingering cost.

**DAG topology** (8 tasks):

1. `trigger_drift` (TriggerDagRunOperator): `trigger_dag_id=f"week22_lab1_{STUDENT_ID}"`, `wait_for_completion=True`, `conf={"force_drift": True}` (force drift for the demo so the gate flips).
2. `trigger_retrain` (TriggerDagRunOperator): `trigger_dag_id=f"week22_lab2_{STUDENT_ID}"`, `wait_for_completion=True`, `conf={"use_pretrained": True}`.
3. `create_model_from_package` (PythonOperator): pull the latest ModelPackageArn from `fraud-classifier-week19`, call `create_model(Containers=[{"ModelPackageName": arn}], ExecutionRoleArn=SAGEMAKER_ROLE_ARN)` with a unique per-student model name, push the ARN to XCom, return the new model name.
4. `create_endpoint_config` (PythonOperator): build a fresh endpoint config naming the new model with `InstanceType="ml.m5.large"`, `InitialInstanceCount=1`. Return the config name.
5. `create_endpoint` (PythonOperator): `create_endpoint(EndpointName=f"fraud-w22-student-NN", EndpointConfigName=...)` - CREATE your own endpoint (not update a shared one).
6. `wait_in_service` (PythonOperator with `execution_timeout=timedelta(minutes=12)`): poll `describe_endpoint` on YOUR endpoint every 30s until `InService`.
7. `verify_new_model` (PythonOperator): four-step lineage trace on your endpoint, invoke it on a known-fraud payload, ask Bedrock Converse to (a) score and (b) cite the ModelPackageArn; assert the cited ARN matches the lineage-traced ARN. Print "LOOP CLOSED".
8. `delete_endpoint` (PythonOperator, `trigger_rule="all_done"`): delete `fraud-w22-student-NN` no matter what happened upstream, so your endpoint never bills past the lab.

**DAG defaults**:
- `dag_id = f"week22_lab3_{STUDENT_ID}"`
- `schedule = None`, `catchup = False`, `start_date = days_ago(1)`
- `default_args = {"retries": 0}`

In [ ]:
# SOLUTION: Lab 3 - orchestrator DAG that closes the loop. The two
# TriggerDagRunOperator tasks were already done. New code:
# create_model_from_package, create_endpoint_config, create_endpoint
# (CREATE your own per-student endpoint, no shared-endpoint update),
# wait_in_service (12-min hard timeout), verify_new_model (lineage
# trace + invoke_endpoint + single Bedrock Converse call), and
# delete_endpoint (trigger_rule="all_done") so your endpoint never
# bills past the lab.

dag_string_3 = f'''
from __future__ import annotations
import json
import time
from datetime import datetime, timedelta, timezone

import boto3
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.trigger_dagrun import TriggerDagRunOperator
from airflow.utils.dates import days_ago

STUDENT_ID = {STUDENT_ID}
ENDPOINT_NAME = "{ENDPOINT_NAME}"
PACKAGE_GROUP = "{PACKAGE_GROUP}"
SAGEMAKER_ROLE_ARN = {SAGEMAKER_ROLE_ARN!r}
BEDROCK_MODEL_ID = "{BEDROCK_MODEL_ID}"

# Each student deploys their OWN endpoint, verifies on it, then deletes it
# at the end. No shared endpoint, no update_endpoint, no 60-way collision.
STUDENT_ENDPOINT = f"fraud-w22-student-{{STUDENT_ID:02d}}"

default_args = {{
    "owner": f"student_{{STUDENT_ID:02d}}",
    "retries": 0,
}}


def _sm():
    return boto3.client("sagemaker")


def _smr():
    return boto3.client("sagemaker-runtime")


def _br():
    return boto3.client("bedrock-runtime")


def _latest_package_arn():
    resp = _sm().list_model_packages(
        ModelPackageGroupName=PACKAGE_GROUP,
        SortBy="CreationTime",
        SortOrder="Descending",
        MaxResults=1,
    )
    return resp["ModelPackageSummaryList"][0]["ModelPackageArn"]


def create_model_from_package(**context):
    arn = _latest_package_arn()
    model_name = f"fraud-w22-{{STUDENT_ID:02d}}-{{int(time.time())}}"
    _sm().create_model(
        ModelName=model_name,
        ExecutionRoleArn=SAGEMAKER_ROLE_ARN,
        Containers=[{{"ModelPackageName": arn}}],
    )
    context["ti"].xcom_push(key="model_package_arn", value=arn)
    return model_name


def create_endpoint_config(**context):
    model_name = context["ti"].xcom_pull(task_ids="create_model_from_package")
    cfg_name = f"fraud-w22-cfg-{{STUDENT_ID:02d}}-{{int(time.time())}}"
    _sm().create_endpoint_config(
        EndpointConfigName=cfg_name,
        ProductionVariants=[{{
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "InstanceType": "ml.m5.large",
            "InitialInstanceCount": 1,
        }}],
    )
    return cfg_name


def create_endpoint(**context):
    # Per-student CREATE (not update). Each student owns STUDENT_ENDPOINT,
    # so there is no cross-student clobber and no need for jitter.
    cfg_name = context["ti"].xcom_pull(task_ids="create_endpoint_config")
    _sm().create_endpoint(
        EndpointName=STUDENT_ENDPOINT,
        EndpointConfigName=cfg_name,
    )
    return STUDENT_ENDPOINT


def wait_in_service(**context):
    deadline = time.time() + 12 * 60
    while time.time() < deadline:
        resp = _sm().describe_endpoint(EndpointName=STUDENT_ENDPOINT)
        status = resp["EndpointStatus"]
        print(f"wait_in_service: status={{status}}")
        if status == "InService":
            return
        if status == "Failed":
            raise RuntimeError(f"Endpoint {{STUDENT_ENDPOINT}} entered Failed state")
        time.sleep(30)
    raise TimeoutError(f"Endpoint {{STUDENT_ENDPOINT}} did not reach InService in 12 min")


def verify_new_model(**context):
    expected_arn = context["ti"].xcom_pull(
        task_ids="create_model_from_package", key="model_package_arn"
    )
    # Four-step lineage trace on YOUR endpoint.
    ep = _sm().describe_endpoint(EndpointName=STUDENT_ENDPOINT)
    cfg = _sm().describe_endpoint_config(EndpointConfigName=ep["EndpointConfigName"])
    mdl = _sm().describe_model(ModelName=cfg["ProductionVariants"][0]["ModelName"])
    traced_arn = mdl["PrimaryContainer"].get("ModelPackageName")
    if traced_arn != expected_arn:
        raise RuntimeError(
            f"Lineage mismatch: traced={{traced_arn}} expected={{expected_arn}}"
        )
    # Invoke the endpoint on a known-fraud-shaped payload.
    payload = {{"amount": 9999.0, "merchant_country": "US_CRYPTO", "card_present": False}}
    inv = _smr().invoke_endpoint(
        EndpointName=STUDENT_ENDPOINT,
        ContentType="application/json",
        Body=json.dumps(payload).encode("utf-8"),
    )
    body = inv["Body"].read().decode("utf-8")
    # Single Bedrock Converse call - no Strands needed.
    prompt = (
        "You are a fraud-classifier supervisor.\\n"
        f"Endpoint prediction: {{body}}\\n"
        f"Model package ARN traced via lineage: {{traced_arn}}\\n"
        "In one sentence: (1) is this transaction fraud, (2) cite the ARN."
    )
    resp = _br().converse(
        modelId=BEDROCK_MODEL_ID,
        messages=[{{"role": "user", "content": [{{"text": prompt}}]}}],
        inferenceConfig={{"maxTokens": 200, "temperature": 0}},
    )
    answer = resp["output"]["message"]["content"][0]["text"]
    print(f"Supervisor answer: {{answer}}")
    if traced_arn not in answer:
        raise RuntimeError("Supervisor did not cite the new ModelPackageArn")
    print(f"LOOP CLOSED. New ModelPackageArn live on {{STUDENT_ENDPOINT}}: {{traced_arn}}")


def delete_endpoint(**context):
    # Always-run cleanup so your per-student endpoint does not bill forever.
    # trigger_rule="all_done" means this runs even if an upstream task failed.
    for name in (STUDENT_ENDPOINT,):
        try:
            _sm().delete_endpoint(EndpointName=name)
            print(f"Deleted endpoint {{name}}")
        except Exception as e:
            print(f"delete_endpoint: {{name}} not deleted ({{e}})")


with DAG(
    dag_id=f"week22_lab3_{{STUDENT_ID}}",
    description="Close-the-loop orchestrator - Week 22 Lab 3",
    default_args=default_args,
    start_date=days_ago(1),
    schedule=None,
    catchup=False,
    tags=["week22", "lab3", f"student_{{STUDENT_ID:02d}}"],
) as dag:
    t_drift = TriggerDagRunOperator(
        task_id="trigger_drift",
        trigger_dag_id=f"week22_lab1_{{STUDENT_ID}}",
        conf={{"force_drift": True}},
        wait_for_completion=True,
        poke_interval=15,
    )
    t_retrain = TriggerDagRunOperator(
        task_id="trigger_retrain",
        trigger_dag_id=f"week22_lab2_{{STUDENT_ID}}",
        conf={{"use_pretrained": True}},
        wait_for_completion=True,
        poke_interval=15,
    )
    t_create_model = PythonOperator(task_id="create_model_from_package",
                                    python_callable=create_model_from_package)
    t_create_cfg   = PythonOperator(task_id="create_endpoint_config",
                                    python_callable=create_endpoint_config)
    t_create_ep    = PythonOperator(task_id="create_endpoint",
                                    python_callable=create_endpoint)
    t_wait         = PythonOperator(task_id="wait_in_service",
                                    python_callable=wait_in_service,
                                    execution_timeout=timedelta(minutes=12))
    t_verify       = PythonOperator(task_id="verify_new_model",
                                    python_callable=verify_new_model)
    t_delete       = PythonOperator(task_id="delete_endpoint",
                                    python_callable=delete_endpoint,
                                    trigger_rule="all_done")

    t_drift >> t_retrain >> t_create_model >> t_create_cfg >> t_create_ep >> t_wait >> t_verify >> t_delete
'''

In [ ]:
# Lab 3 - upload, poll, trigger. The orchestrator DAG fires Lab 1,
# waits, fires Lab 2, waits, then drives the endpoint update +
# agentic verification. Total wall time about 7-10 min, dominated by
# wait_in_service (the endpoint update is the slow part).
#
# Open the Airflow UI in a second tab while this runs. You will see
# three DAGs lit up: the orchestrator (this one), and Lab 1 + Lab 2
# under its "Triggered DAG Runs" links.

lab3_dag_id = f"week22_lab3_{STUDENT_ID}"

upload_dag(STUDENT_ID, 3, dag_string_3)
wait_for_dag_pickup(lab3_dag_id)

# Lab 3 uses TriggerDagRunOperator to fire Lab 1 and Lab 2 from inside the
# DAG. A TriggerDagRunOperator pointed at a PAUSED dag creates a run that
# stays queued forever, so unpause both sub-DAGs first (idempotent).
unpause_dag(f"week22_lab1_{STUDENT_ID}")
unpause_dag(f"week22_lab2_{STUDENT_ID}")

run_lab3 = trigger_dag(lab3_dag_id)
# Generous timeout - 15 min covers the worst-case endpoint update queue.
wait_for_dag_run(lab3_dag_id, run_lab3, timeout=900)

print("\nLab 3 orchestrator DAG run finished. Check the verify_new_model "
      "task log for the LOOP CLOSED line.")


In [ ]:
# Final notebook-level sanity check on top of the DAG-level one.
# Your Lab 3 DAG created (and then deleted) fraud-w22-student-NN. We
# confirm the loop closed by checking the registry's latest version is
# the one your create_model_from_package task pulled. The per-student
# endpoint itself is already deleted by the all_done cleanup task, so we
# assert on the registry + the task log rather than re-describing it.

latest_resp = sagemaker_client.list_model_packages(
    ModelPackageGroupName=PACKAGE_GROUP,
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=1,
)
latest_arn = latest_resp["ModelPackageSummaryList"][0]["ModelPackageArn"]

print("Latest model package in the registry:")
print(f"  {latest_arn}")
print()
print("=" * 60)
print("LOOP CLOSED")
print("Your Lab 3 DAG: drift -> retrain -> register -> create your own")
print("endpoint -> verify the new model on it -> delete it.")
print("Look for the 'LOOP CLOSED' line in the verify_new_model task log")
print("in the Airflow UI for the agentic confirmation.")
print("=" * 60)

## Think About It - Lab 3 reflection

1. Sixty students run Lab 3 in the same 2-hour window. In this design each one creates their OWN endpoint, `fraud-w22-student-NN`, instead of all updating one shared endpoint. What problem does that avoid? (Hint: SageMaker serializes updates per endpoint, so a shared endpoint forces students into a queue and most get a `ValidationException`.) What is the cost trade-off of one-endpoint-per-student, and how does the `delete_endpoint` cleanup task keep that cost bounded?

2. The `delete_endpoint` task has `trigger_rule="all_done"`. Why is that the right trigger rule for a cleanup task, rather than the default `all_success`? Think about what happens to your endpoint (and your bill) if `verify_new_model` fails and the cleanup only ran on success.

3. `default_args` keeps `retries=0`. Why do we NOT retry the deploy tasks here? Think about what happens if `create_endpoint` succeeded but the task instance crashed before marking itself green, and a retry then tries to create the same-named endpoint again.

## Where this goes next

The production-craft topics you saw in the pre-class videos - Celery vs KEDA executor comparison, AutoDAG / dynamic DAG generation, catchup and backfilling - are intentionally NOT in today's labs. They are Week 24 capstone material. You have vocabulary parity with the videos but no hands-on time today because closing the drift -> retrain -> redeploy loop is the harder, more valuable skill for this point in the program.

There is an **optional deep-dive notebook** in the same folder (`week_22_optional_dag_authoring.ipynb`) that walks through DAG-authoring topics you do NOT need for class: dynamic task mapping with `@task.expand`, deferrable operators with `defer_to_trigger`, sensor poking strategies, and TaskGroup composition. It is self-contained and supplementary; skip it if you are happy with what you built today.


## Glossary - new Airflow primitives this week

| Primitive | Purpose |
|---|---|
| `ShortCircuitOperator` | Skip downstream tasks when a Python callable returns False. |
| `ignore_downstream_trigger_rules=False` | On ShortCircuit: let cleanup tasks with `trigger_rule="all_done"` still run. |
| `BranchPythonOperator` | Pick which downstream branch to take by returning a task_id (or list). |
| `TriggerDagRunOperator` | One DAG triggers another, optionally waiting for completion. |
| `SageMakerTrainingOperator` | Wraps `create_training_job`; `check_if_job_exists` + timestamp action protect reruns. |
| `SageMakerRegisterModelVersionOperator` | The canonical model-package registration operator on provider 9.0.0. |
| `SageMakerTrainingSensor` | Sensor wrapper for `describe_training_job`; raises on terminal failure. |
| `trigger_rule="all_done"` | Run regardless of upstream success/failure/skip. |
| `trigger_rule="none_failed_min_one_success"` | Run when no upstream failed AND at least one succeeded (use with branches). |
| `wait_for_completion=True` | Block this task until the wrapped operation finishes. |
| `execution_timeout=timedelta(minutes=N)` | Hard cap on task wall time; raises AirflowTaskTimeout. |

## Closing

This week you stopped being a DAG operator and became a DAG author. The retrain loop is now yours - drift in, retrain, register, redeploy, verify. The Bedrock supervisor reads back the new ModelPackageArn from lineage and you wrote every line of code that got it there.

In Week 23 we shift to AI ethics on top of this same fraud system. In Week 24 you take everything from Weeks 1 through 23 into the capstone.
